# EDA AI4I 2020 — Predictive Maintenance Dataset
Sumber data: UCI ML Repository id 601 (CC BY 4.0), Matzka (2020). File: `ai4i2020.csv`.

## Cara pakai
- **Kaggle:** tambahkan dataset `ai4i2020.csv` via *Add Input*, lalu ubah `DATA_PATH` di sel berikut sesuai path input (contoh: `/kaggle/input/<nama-dataset>/ai4i2020.csv`). Atau upload manual ke working directory dan biarkan deteksi otomatis.
- **Google Colab:** upload `ai4i2020.csv` ke *Files* panel, atau biarkan notebook meminta upload otomatis bila file tidak ditemukan.
- **Lokal:** letakkan `ai4i2020.csv` di folder yang sama dengan notebook ini.

Hanya butuh `pandas`, `numpy`, `matplotlib` (sudah tersedia di Kaggle/Colab). Warna konsisten: **gagal = merah**, **tidak gagal = biru**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FAIL_C = "#D32F2F"  # merah = gagal
OK_C = "#1976D2"    # biru = tidak gagal
OUTPUT_DIR = "."  # PNG hasil disimpan di sini

In [ ]:
# === 1. Lokasi data (EDIT bila perlu) ===
DATA_PATH = "ai4i2020.csv"

candidates = [
    DATA_PATH,
    "ai4i2020.csv",
    "data/ai4i2020.csv",
    "/kaggle/input/ai4i-2020-predictive-maintenance-dataset/ai4i2020.csv",
    r"C:\Users\Afif\Downloads\ai4i+2020+predictive+maintenance+dataset\ai4i2020.csv",
]
SRC = next((p for p in candidates if os.path.exists(p)), None)
if SRC is None:
    try:
        from google.colab import files  # hanya ada di Colab
        print("File tidak ditemukan. Silakan upload ai4i2020.csv.")
        uploaded = files.upload()
        SRC = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            "ai4i2020.csv tidak ditemukan. Upload file atau perbaiki DATA_PATH."
        )
print("Memakai data:", SRC)
df = pd.read_csv(SRC)
print("shape:", df.shape)
print("missing total:", int(df.isna().sum().sum()))
print("duplikat baris:", int(df.duplicated().sum()))

In [ ]:
# === 2. Profil singkat ===
print("target:", df["Machine failure"].value_counts().to_dict(),
      "| rate gagal: %.2f%%" % (df["Machine failure"].mean() * 100))
print("mode:", {c: int(df[c].sum()) for c in ["TWF", "HDF", "PWF", "OSF", "RNF"]})
print("tipe:", df["Type"].value_counts().to_dict())
print(pd.crosstab(df["Type"], df["Machine failure"]).to_string())
print("rate gagal per tipe (%):",
      (df.groupby("Type")["Machine failure"].mean() * 100).round(2).to_dict())
num = ["Air temperature [K]", "Process temperature [K]",
       "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
display(df[num].describe().round(2))
or_flags = (df[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum(axis=1) > 0).astype(int)
print("label != OR(flag):", int((or_flags != df["Machine failure"]).sum()))

In [ ]:
# === 3. Histogram sebaran fitur numerik (gagal vs tidak gagal) ===
ok = df[df["Machine failure"] == 0]
bad = df[df["Machine failure"] == 1]
titles = {"Air temperature [K]": "Suhu Udara [K]",
          "Process temperature [K]": "Suhu Proses [K]",
          "Rotational speed [rpm]": "Kecepatan Putar [rpm]",
          "Torque [Nm]": "Torsi [Nm]",
          "Tool wear [min]": "Keausan Pahat [mnt]"}
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()
for i, c in enumerate(num):
    ax = axes[i]
    ax.hist(ok[c], bins=40, color=OK_C, alpha=0.6, label="Tidak gagal (9661)")
    ax.hist(bad[c], bins=40, color=FAIL_C, alpha=0.7, label="Gagal (339)")
    ax.set_title("Sebaran " + titles[c])
    ax.set_xlabel(titles[c]); ax.set_ylabel("Jumlah baris")
    ax.legend(fontsize=9)
axes[5].axis("off")
fig.suptitle("Sebaran Fitur Numerik berdasarkan Status Kegagalan Mesin", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(os.path.join(OUTPUT_DIR, "01_histogram_fitur_numerik.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# === 4. Boxplot per fitur per target ===
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for ax, c in zip(axes, num):
    bp = ax.boxplot([ok[c].values, bad[c].values], tick_labels=["Tidak\ngagal", "Gagal"],
                    patch_artist=True)
    bp["boxes"][0].set_facecolor(OK_C); bp["boxes"][1].set_facecolor(FAIL_C)
    for box in bp["boxes"]:
        box.set_alpha(0.6)
    ax.set_title(titles[c])
fig.suptitle("Boxplot Fitur Numerik per Status Kegagalan", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.88])
fig.savefig(os.path.join(OUTPUT_DIR, "02_boxplot_fitur_per_target.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# === 5. Proporsi gagal vs tidak gagal ===
counts = df["Machine failure"].value_counts().sort_index()
labels = ["Tidak gagal", "Gagal"]
vals = [int(counts.get(0, 0)), int(counts.get(1, 0))]
pct = [v / len(df) * 100 for v in vals]
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(labels, vals, color=[OK_C, FAIL_C], alpha=0.85)
for b, v, p in zip(bars, vals, pct):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
            f"{v:,}\n({p:.2f}%)".replace(",", "."), ha="center", va="bottom", fontsize=12)
ax.set_title("Proporsi Mesin Gagal vs Tidak Gagal (n=10.000)", fontsize=13, fontweight="bold")
ax.set_ylabel("Jumlah baris")
ax.set_ylim(0, max(vals) * 1.18)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "03_proporsi_gagal.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# === 6. Per tipe produk: jumlah + tingkat kegagalan ===
ct = pd.crosstab(df["Type"], df["Machine failure"])
for col in [0, 1]:
    if col not in ct.columns:
        ct[col] = 0
ct = ct[[0, 1]].reindex(["L", "M", "H"])
rate = ct[1] / ct.sum(axis=1) * 100
x = np.arange(len(ct.index)); w = 0.35
fig, ax1 = plt.subplots(figsize=(9, 6))
ax1.bar(x - w / 2, ct[0].values, w, color=OK_C, alpha=0.85, label="Tidak gagal")
ax1.bar(x + w / 2, ct[1].values, w, color=FAIL_C, alpha=0.9, label="Gagal")
for i in range(len(ct.index)):
    ax1.text(i - w / 2, ct[0].values[i], f"{ct[0].values[i]:,}".replace(",", "."),
             ha="center", va="bottom")
    ax1.text(i + w / 2, ct[1].values[i], f"{ct[1].values[i]:,}".replace(",", "."),
             ha="center", va="bottom")
ax1.set_xticks(x); ax1.set_xticklabels([f"{t} (n={ct.sum(axis=1)[t]:,})".replace(",", ".") for t in ct.index])
ax1.set_ylabel("Jumlah baris"); ax1.set_xlabel("Tipe produk")
ax1.set_title("Jumlah dan Tingkat Kegagalan per Tipe Produk", fontsize=13, fontweight="bold")
ax2 = ax1.twinx()
ax2.plot(x, rate.values, color="#333333", marker="o", linewidth=2, label="Tingkat gagal (%)")
for i, v in enumerate(rate.values):
    ax2.text(i, v, f" {v:.2f}%".replace(".", ","), va="bottom")
ax2.set_ylabel("Tingkat kegagalan (%)")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "04_per_tipe_produk.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# === 7. Heatmap korelasi fitur numerik + target ===
corr = df[num + ["Machine failure"]].corr()
print("Korelasi vs gagal:", corr["Machine failure"].round(3).to_dict())
short = {"Air temperature [K]": "Suhu udara", "Process temperature [K]": "Suhu proses",
         "Rotational speed [rpm]": "Kecepatan", "Torque [Nm]": "Torsi",
         "Tool wear [min]": "Keausan", "Machine failure": "Gagal"}
order = num + ["Machine failure"]
tick = [short[c] for c in order]
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.matshow(corr.loc[order, order].values, cmap="RdYlBu_r", vmin=-1, vmax=1)
for i in range(len(order)):
    for j in range(len(order)):
        v = corr.loc[order, order].values[i, j]
        ax.text(j, i, f"{v:.2f}".replace(".", ","), ha="center", va="center",
                color="white" if abs(v) > 0.5 else "black")
ax.set_xticks(range(len(order))); ax.set_yticks(range(len(order)))
ax.set_xticklabels(tick, rotation=30, ha="left"); ax.set_yticklabels(tick)
ax.set_title("Heatmap Korelasi Fitur Numerik dan Kegagalan", pad=28, fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, label="Korelasi Pearson")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "05_heatmap_korelasi.png"), dpi=200, bbox_inches="tight")
plt.show()

## Ringkas baca hasil
- 339 gagal (3,39%) → akurasi menyesatkan; pakai presisi/recall/F1/PR-AUC.
- Tingkat gagal per tipe: L 3,92%, M 2,77%, H 2,09%.
- Korelasi linear vs target lemah (tertinggi torsi 0,19) → hubungan berbentuk ambang/kombinasi, bukan linear.
- Pasangan berkorelasi kuat: suhu udara–proses (0,88), kecepatan–torsi (−0,88).
- Detail angka dan keterbatasan ada di `RINGKASAN_TEMUAN.md`.